In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv("train_transaction.csv")
test  = pd.read_csv("test_transaction.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())

In [ ]:
y = train["isFraud"]  # target
X = train.drop(columns=["isFraud", "TransactionID"])

# Drop TransactionID from test as well
test_features = test.drop(columns=["TransactionID"])


In [ ]:
# Split before fitting any learned preprocessing to avoid validation leakage.
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


In [ ]:
# Infer column groups from the training split only. Ordinal encoding keeps
# output width fixed; unseen validation/test categories map to -1. These codes
# are compact tree inputs, not categories with a natural or economic order.
categorical_columns = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_columns = X_train.columns.difference(categorical_columns).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1,
                dtype=np.float32,
            )),
        ]), categorical_columns),
    ],
    remainder="drop",
)
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
test_processed = preprocessor.transform(test_features)


Fraud is the minority class, so accuracy can look high even when fraud detection is poor. ROC-AUC and average precision assess ranking; precision, recall, F1, and the confusion matrix below describe predictions at the explicit default 0.5 threshold. The threshold is not optimized for an operational cost trade-off. These are local holdout metrics, not Kaggle leaderboard results.


In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",
    n_jobs=-1,
    eval_metric="auc"
)

In [ ]:
model.fit(
    X_train_processed, y_train,
    eval_set=[(X_val_processed, y_val)],
    verbose=50
)


## Validation metric context

ROC-AUC is computed on the held-out 20% stratified validation split, not on the competition leaderboard. The split uses `random_state=42`. Preprocessing is fitted only on the training partition and reused unchanged for validation and competition test rows. The result depends on the provided competition data and installed dependency versions; this repository does not include the data.


In [ ]:
val_preds = model.predict_proba(X_val_processed)[:, 1]
val_pred_labels = (val_preds >= 0.5).astype(int)
print("Held-out validation ROC-AUC:", roc_auc_score(y_val, val_preds))
print("Held-out validation average precision:", average_precision_score(y_val, val_preds))
print("Precision at 0.5:", precision_score(y_val, val_pred_labels, zero_division=0))
print("Recall at 0.5:", recall_score(y_val, val_pred_labels, zero_division=0))
print("F1 at 0.5:", f1_score(y_val, val_pred_labels, zero_division=0))
print("Confusion matrix at 0.5 (rows=true, columns=predicted):\n", confusion_matrix(y_val, val_pred_labels))


In [ ]:
# Transform test rows with the same preprocessor fitted on X_train.
test_preds = model.predict_proba(test_processed)[:, 1]


In [ ]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_preds
})

submission.to_csv("submission.csv", index=False)
print("Submission saved: submission.csv")